## Importing libraries


In [ ]:
# Import Polars for dataframe operations
import polars as pl

# Import Path for managing file paths
from pathlib import Path

## Define Data folder


In [ ]:
# Define path to data folder
data_folder = Path("../data")

## Load Cleaned Dataset (Parquet Version)


In [ ]:
# Load cleaned sales dataset
sales_df = pl.read_parquet(data_folder / "sales_cleaned.parquet")

# Check data types
print(sales_df.schema)

# preview data set
sales_df.head()

print("Rows:", sales_df.height)
print("Columns:", sales_df.width)

## Find Latest Purchase Date


In [ ]:
# Find the most recent purchase date
latest_date = sales_df.select(pl.col("purchase_date").max()).item()
latest_date

## Calculate Recency

Recency = Days since last purchase


In [ ]:
# Get last purchase date per customer
receny_df = sales_df.group_by("customer_id").agg(
    pl.col("purchase_date").max().alias("last_purchase")
)

# Calculate recency

receny_df = receny_df.with_columns(
    (latest_date - pl.col("last_purchase")).dt.total_days().alias("recency")
)

receny_df.head()

## Calculate Frequency

Frequency = number of purchases


In [ ]:
# Count purchases per customer

frequency_df = sales_df.group_by("customer_id").agg(
    pl.count().alias("frequency")
)

frequency_df.head()

## Calculate Monetary Value

Monetary = total revenue spent


In [ ]:
# Calculate total revenue per customer

monetary_df = sales_df.group_by("customer_id").agg(
    pl.col("revenue").sum().alias("monetary")
)

monetary_df.head()

## Merge RFM Metrics


In [ ]:
# Merge recency, frequency and monetary tables
rfm_df = receny_df.join(frequency_df, on="customer_id").join(
    monetary_df, on="customer_id"
)

rfm_df.head()

## Create RFM Scores


In [ ]:
# Create RFM scores using rank

rfm_df = rfm_df.with_columns(
    [
        pl.col("recency").rank("dense").alias("r_score"),
        pl.col("frequency").rank("dense").alias("f_score"),
        pl.col("monetary").rank("dense").alias("m_score"),
    ]
)

rfm_df.head()

## Create RFM Score


In [ ]:
# Combine RFM scores

rfm_df = rfm_df.with_columns(
    (
        pl.col("r_score") +
        pl.col("f_score") +
        pl.col("m_score")
    )
    .alias("rfm_score")
)

rfm_df.head()

## Create Customer Segments


In [ ]:
# Assign customer segments

rfm_df = rfm_df.with_columns(
    pl.when(pl.col("rfm_score") >= 1500).then(pl.lit("Champions"))
    .when(pl.col("rfm_score") >= 1200).then(pl.lit("Loyal Customers"))
    .when(pl.col("rfm_score") >= 600).then(pl.lit("Potential Loyalists"))
    .otherwise(pl.lit("At Risk"))
    .alias("segment")
)

rfm_df.head()

## Segmet Distribution


In [ ]:
# Count customers per segment

rfm_df.group_by("segment").count().sort("count", descending=True)

## Save Dataset


In [ ]:
# Save RFM dataframe as csv
rfm_df.write_csv(data_folder / "customer_rfm.csv")

In [ ]:
# Save RFM dataframe as parquet
rfm_df.write_parquet(data_folder / "customer_rfm.parquet")

# Sanity Check

In [ ]:
rfm_df.describe()

In [ ]:
rfm_df.group_by("segment").count()